In [2]:
import pandas as pd 
import numpy as np 
import statsmodels.formula.api as smf 
from copy import deepcopy 
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error
import ast 

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt 
import seaborn as sns 
from IPython.display import display

import importlib
#import pgg_helper as pgg 


import warnings
warnings.filterwarnings("ignore")

In [3]:
processed_data_dir = "../data/processed_data/"
df_paired_val = pd.read_csv(processed_data_dir + "df_paired_val.csv")
df_paired_learn = pd.read_csv(processed_data_dir + "df_paired_learn.csv")
df_analysis_val = pd.read_csv(processed_data_dir + "df_analysis_val.csv")
df_analysis_learn = pd.read_csv(processed_data_dir + "df_analysis_learn.csv")
df_rounds_learn = pd.read_csv(processed_data_dir + "df_rounds_learn.csv")
df_rounds_val = pd.read_csv(processed_data_dir + "df_rounds_val.csv")
df_predictions = pd.read_csv(processed_data_dir + "prediction_survey.csv").query("prediction.between(-0.2,1.2)")
df_learning_config = pd.read_csv("../data/exp_config_files/learning.csv")
df_validation_config = pd.read_csv("../data/exp_config_files/validation.csv")

df_rounds_learn = df_rounds_learn.merge(df_analysis_learn, on="gameId", how="left").query("valid_number_of_starting_players == True").dropna(subset=["data.roundPayoff"]).reset_index(drop=True)
df_rounds_val = df_rounds_val.merge(df_analysis_val, on="gameId", how="left").query("valid_number_of_starting_players == True").dropna(subset=["data.roundPayoff"]).reset_index(drop=True)

# Main Text Statistics  

## General 

In [4]:
# Check condition overlap 
print(df_learning_config[["playerCount", "numRounds", "showNRounds", "multiplier", "allOrNothing", "chat"]].shape, df_validation_config[["playerCount", "numRounds", "showNRounds", "multiplier", "allOrNothing", "chat"]].shape)
print(pd.concat([df_learning_config, df_validation_config], ignore_index=True).drop_duplicates(subset=["playerCount", "numRounds", "showNRounds", "multiplier", "allOrNothing", "chat", "punishmentExists"]).shape)

(512, 6) (40, 6)
(552, 24)


In [5]:
# Number of experimental conditions 
df_analysis_learn.query("valid_number_of_starting_players")["name"].nunique() + df_analysis_val.query("valid_number_of_starting_players")["name"].nunique()

360

In [6]:
# Number of players who have made at least 1 decision
df_rounds_learn["playerId"].nunique() + df_rounds_val["playerId"].nunique()

7100

In [7]:
# Number of contributions 
len(df_rounds_learn) + len(df_rounds_val) 

102343

In [8]:
# Number of decisions made to reward/punish others, aggregated across learning and validation waves 

(df_rounds_learn
.assign(n_punishment_decisions = lambda x: [len([k for k,v in ast.literal_eval(y).items() if v > 0]) for y in x["data.punished"]])
.assign(n_reward_decisions = lambda x: [len([k for k,v in ast.literal_eval(y).items() if v > 0]) for y in x["data.rewarded"]])).filter(like="_decisions").sum() +\
(df_rounds_val
.assign(n_punishment_decisions = lambda x: [len([k for k,v in ast.literal_eval(y).items() if v > 0]) for y in x["data.punished"]])
.assign(n_reward_decisions = lambda x: [len([k for k,v in ast.literal_eval(y).items() if v > 0]) for y in x["data.rewarded"]])).filter(like="_decisions").sum()

n_punishment_decisions    15471
n_reward_decisions        29804
dtype: int64

## Average Effects and Heterogeneity Analysis
* Regressions are in R 

In [9]:
print("Average contributions by treatment in learning:\n", df_rounds_learn.groupby("CONFIG_punishmentExists")["data.contribution"].mean() / 20)
print("\n")
print("Average contributions by treatment in validation:\n", df_rounds_val.groupby("CONFIG_punishmentExists")["data.contribution"].mean() / 20)

Average contributions by treatment in learning:
 CONFIG_punishmentExists
False    0.730459
True     0.801636
Name: data.contribution, dtype: float64


Average contributions by treatment in validation:
 CONFIG_punishmentExists
False    0.736287
True     0.822916
Name: data.contribution, dtype: float64


In [10]:
print("Average normalized efficiency by treatment in learning:\n", df_analysis_learn.query("valid_number_of_starting_players").groupby("CONFIG_punishmentExists")["itt_relative_efficiency"].mean())
print("\n")
print("Average normalized efficiency by treatment in validation:\n", df_analysis_val.query("valid_number_of_starting_players").groupby("CONFIG_punishmentExists")["itt_relative_efficiency"].mean())

Average normalized efficiency by treatment in learning:
 CONFIG_punishmentExists
False    0.714319
True     0.625333
Name: itt_relative_efficiency, dtype: float64


Average normalized efficiency by treatment in validation:
 CONFIG_punishmentExists
False    0.719745
True     0.676769
Name: itt_relative_efficiency, dtype: float64


In [11]:
# Heterogeneity in clustered learning wave experiments 
CLUSTER_COLUMNS = ['CONFIG_playerCount', 'CONFIG_numRounds', 'CONFIG_showNRounds', 
                   'CONFIG_MPCR', 'CONFIG_allOrNothing', 'CONFIG_chat', 'CONFIG_defaultContribProp', 
                   'CONFIG_rewardExists', 'CONFIG_showOtherSummaries', "CONFIG_punishmentCost", "CONFIG_punishmentTech",
                   'CONFIG_showPunishmentId']

df_analysis_learn_kmeans = deepcopy(df_analysis_learn)
df_analysis_learn_kmeans.loc[:,["CONFIG_playerCount", "CONFIG_numRounds", "CONFIG_punishmentCost", "CONFIG_punishmentTech"]] = MinMaxScaler().fit_transform(df_analysis_learn_kmeans.loc[:,["CONFIG_playerCount", "CONFIG_numRounds", "CONFIG_punishmentCost", "CONFIG_punishmentTech"]])

kmeans = KMeans(n_clusters=20, random_state=2024)
df_analysis_learn_kmeans["cluster"] = kmeans.fit_predict(df_analysis_learn_kmeans[CLUSTER_COLUMNS])
df_cluster_estimates = df_analysis_learn_kmeans.query("valid_number_of_starting_players").groupby("cluster").apply(lambda x: smf.ols("itt_relative_efficiency ~ CONFIG_punishmentExists", data=x).fit()).reset_index().rename(columns={0:"ols_model"})
df_cluster_estimates["treatment_effect_mean"] = df_cluster_estimates["ols_model"].apply(lambda x: x.params["CONFIG_punishmentExists[T.True]"])
df_cluster_estimates["treatment_effect_se"] = df_cluster_estimates["ols_model"].apply(lambda x: x.bse["CONFIG_punishmentExists[T.True]"])

print("Heterogeneity measures in learning wave clusters:\n")
print(pgg.analysis.calc_q_i2(df_cluster_estimates[["treatment_effect_mean", "treatment_effect_se"]]))

Heterogeneity measures in learning wave clusters:



NameError: name 'pgg' is not defined

In [ ]:
# Heterogeneity in the validation wave experiments 
df_paired_val["ols_model"] = df_paired_val["CONFIG_configId"].map(df_analysis_val.query("valid_number_of_starting_players").groupby("CONFIG_configId").apply(lambda x: smf.ols("itt_relative_efficiency ~ CONFIG_punishmentExists", data=x).fit()))
df_paired_val["treatment_effect_mean"] = df_paired_val["ols_model"].apply(lambda x: x.params["CONFIG_punishmentExists[T.True]"])
df_paired_val["treatment_effect_se"] = df_paired_val["ols_model"].apply(lambda x: x.bse["CONFIG_punishmentExists[T.True]"])

print("Heterogeneity measures in validation wave experiments:\n")
print(pgg.analysis.calc_q_i2(df_paired_val[["treatment_effect_mean", "treatment_effect_se"]]))

Heterogeneity measures in validation wave experiments:

{'Q': 30.29, 'Q_pval': 0.048, 'Q_dof': 19, 'i2': 0.37}


In [ ]:
# Heterogeneity when aggregating learning clusters and validation experiments 
print(pgg.analysis.calc_q_i2(pd.concat([df_cluster_estimates, df_paired_val], ignore_index=True)[["treatment_effect_mean", "treatment_effect_se"]]))

{'Q': 59.34, 'Q_pval': 0.019, 'Q_dof': 39, 'i2': 0.34}


In [ ]:
# Min and max effect sizes in learning and validation waves 
min_effect_learn_cluster, max_effect_learn_cluster = df_cluster_estimates.sort_values("treatment_effect_mean")["cluster"].values[[0,-1]]
min_effect_val_exp, max_effect_val_exp = df_paired_val.sort_values("treatment_effect_mean")["CONFIG_configId"].values[[0,-1]]

print("Min and max punishment effect clusters in learning wave: \n")
print(df_analysis_learn_kmeans.query("valid_number_of_starting_players").groupby(["cluster", "CONFIG_punishmentExists"])["itt_relative_efficiency"].mean()[[min_effect_learn_cluster, max_effect_learn_cluster]])
print("\n")
print("Min and max punishment effect experiments in validation wave: \n")
print(df_analysis_val.query("valid_number_of_starting_players").groupby(["CONFIG_configId", "CONFIG_punishmentExists"])["itt_relative_efficiency"].mean()[[min_effect_val_exp, max_effect_val_exp]])

Min and max punishment effect clusters in learning wave: 

cluster  CONFIG_punishmentExists
10       False                      0.780100
         True                       0.439470
2        False                      0.555775
         True                       0.802144
Name: itt_relative_efficiency, dtype: float64


Min and max punishment effect experiments in validation wave: 

CONFIG_configId  CONFIG_punishmentExists
1                False                      0.717841
                 True                       0.397473
3                False                      0.625639
                 True                       0.805342
Name: itt_relative_efficiency, dtype: float64


## Model Predictions and Forecasting Survey Analysis 


In [12]:
def bootstrap_model_evaluation(prediction_column, df_validation_sample):    
    rmse = np.sqrt(mean_squared_error(y_true=df_validation_sample["treatment_itt_efficiency"]*100, y_pred=df_validation_sample[prediction_column]*100))
    return rmse

In [13]:
df_paired_val["woc_sspp_pred"] = df_paired_val["CONFIG_configId"].map(df_predictions.query("source == 'sspp'").groupby("CONFIG_configId")["prediction"].mean())
df_paired_val["woc_prolific_pred"] = df_paired_val["CONFIG_configId"].map(df_predictions.query("source == 'prolific'").groupby("CONFIG_configId")["prediction"].mean())
df_paired_val["baseline"] = df_paired_learn["treatment_itt_efficiency"].mean()

In [15]:
for model in ["elastic_prereg", "ols_prereg", "rf_prereg", "xgb_prereg", "mlp_prereg", "woc_sspp", "woc_prolific"]:
    model_rmse = np.sqrt(mean_squared_error(df_paired_val["treatment_itt_efficiency"]*100, df_paired_val[f"{model}_pred"]*100))
    model_r2 = 1 - mean_squared_error(df_paired_val["treatment_itt_efficiency"]*100, df_paired_val[f"{model}_pred"]*100) / mean_squared_error(df_paired_val["treatment_itt_efficiency"]*100, df_paired_val["baseline"]*100)
    print(f"{model}: RMSE = {model_rmse}, R2 = {model_r2}")

elastic_prereg: RMSE = 4.516374610310278, R2 = 0.5333430997709674
ols_prereg: RMSE = 5.222765062410205, R2 = 0.3759509047654118
rf_prereg: RMSE = 5.399124755341142, R2 = 0.33309418734776264
xgb_prereg: RMSE = 6.326798725589932, R2 = 0.08423126077558607
mlp_prereg: RMSE = 5.786596814228069, R2 = 0.2339374627059485
woc_sspp: RMSE = 6.534930517567188, R2 = 0.022988386685350415
woc_prolific: RMSE = 6.435945225446866, R2 = 0.05236202516429367


In [16]:
mean_squared_error(df_paired_val["treatment_itt_efficiency"]*100, df_paired_val["baseline"]*100)

43.710142527934906

In [ ]:
dict_model_perf_bootstrap = {}

for model_label in ["ols_prereg_pred","rf_prereg_pred","xgb_prereg_pred","mlp_prereg_pred","elastic_prereg_pred", "woc_prolific_pred", "woc_sspp_pred", "baseline"]:
    dict_model_perf_bootstrap[model_label] = [bootstrap_model_evaluation(model_label, df_paired_val.sample(n=20, replace=True, random_state=x)) for x in range(1000)]
    
df_bootstrap = pd.DataFrame(dict_model_perf_bootstrap)
for model in ["ols", "rf", "xgb", "mlp", "elastic"]:
    df_bootstrap[f"{model}_baseline_rmse_ratio"] = df_bootstrap[f"{model}_prereg_pred"] / df_bootstrap["baseline"]
    df_bootstrap[f"{model}_baseline_rmse_diff"] = df_bootstrap["baseline"] - df_bootstrap[f"{model}_prereg_pred"] 
    df_bootstrap[f"{model}_woc_sspp_rmse_ratio"] = df_bootstrap[f"{model}_prereg_pred"] / df_bootstrap["woc_sspp_pred"]
    df_bootstrap[f"{model}_woc_prolific_rmse_ratio"] = df_bootstrap[f"{model}_prereg_pred"] / df_bootstrap["woc_prolific_pred"]
    df_bootstrap[f"{model}_woc_sspp_rmse_diff"] = df_bootstrap["woc_sspp_pred"] - df_bootstrap[f"{model}_prereg_pred"] 
    df_bootstrap[f"{model}_woc_prolific_rmse_diff"] = df_bootstrap["woc_prolific_pred"] - df_bootstrap[f"{model}_prereg_pred"] 
    
df_bootstrap["woc_prolific_baseline_rmse_ratio"] = df_bootstrap["woc_prolific_pred"] / df_bootstrap["baseline"]
df_bootstrap["woc_sspp_baseline_rmse_ratio"] = df_bootstrap["woc_sspp_pred"] / df_bootstrap["baseline"]
df_bootstrap["woc_sspp_minus_prolific"] = df_bootstrap["woc_sspp_pred"] - df_bootstrap["woc_prolific_pred"]

In [ ]:
df_bootstrap.apply(lambda x: x.quantile([0.025,0.975]), axis=0).round(2).T

,0.025,0.975
ols_prereg_pred,3.93,6.52
rf_prereg_pred,3.55,7.39
xgb_prereg_pred,4.19,8.52
mlp_prereg_pred,4.16,7.33
elastic_prereg_pred,3.35,5.69
woc_prolific_pred,4.59,8.08
woc_sspp_pred,4.61,8.77
baseline,5.15,7.91
ols_baseline_rmse_ratio,0.57,1.14
ols_baseline_rmse_diff,-0.70,3.12


## Feature importance 

In [ ]:
PREDICTION_FEATURE_COLS = pgg.constants.PREDICTION_FEATURE_COLS

In [ ]:
dict_unfitted_models = pgg.analysis.get_models(hpo_json_filepath="../data/hpo_model_configs.json", 
                                        df_paired_learn=df_paired_learn,
                                        feature_cols=PREDICTION_FEATURE_COLS,
                                        target_col="treatment_itt_efficiency",
                                        fitted=False)
    
oos_ols_featimp_prereg = pgg.analysis.oos_permutation_feature_importance(dict_unfitted_models["ols"], df_paired_learn, df_paired_val, PREDICTION_FEATURE_COLS, "treatment_itt_efficiency", "OLS", n_iterations=30)
oos_rf_featimp_prereg = pgg.analysis.oos_permutation_feature_importance(dict_unfitted_models["rf"], df_paired_learn, df_paired_val, PREDICTION_FEATURE_COLS, "treatment_itt_efficiency", "RF", n_iterations=30)
oos_xgb_featimp_prereg = pgg.analysis.oos_permutation_feature_importance(dict_unfitted_models["xgb"], df_paired_learn, df_paired_val, PREDICTION_FEATURE_COLS, "treatment_itt_efficiency", "XGB", n_iterations=30)
oos_mlp_featimp_prereg = pgg.analysis.oos_permutation_feature_importance(dict_unfitted_models["mlp"], df_paired_learn, df_paired_val, PREDICTION_FEATURE_COLS, "treatment_itt_efficiency", "MLP", n_iterations=30)
oos_enet_featimp_prereg = pgg.analysis.oos_permutation_feature_importance(dict_unfitted_models["enet"], df_paired_learn, df_paired_val, PREDICTION_FEATURE_COLS, "treatment_itt_efficiency", "E-net", n_iterations=30)

oos_master_featimp_prereg = pd.concat([oos_ols_featimp_prereg, oos_xgb_featimp_prereg, oos_enet_featimp_prereg, oos_rf_featimp_prereg, oos_mlp_featimp_prereg], ignore_index = True)

oos_master_featimp_prereg["feature"] = oos_master_featimp_prereg["feature"].map({'CONFIG_playerCount':"# of players", 
'CONFIG_numRounds':"# of rounds",
'CONFIG_showNRounds':"Visibility of # of rounds", 
'CONFIG_MPCR':"Marginal per capita return",
'CONFIG_allOrNothing':'"All or Nothing" contributions', 
'CONFIG_chat':"Ability to chat", 
'CONFIG_defaultContribProp':"Default contribution vs withdrawal",
'CONFIG_rewardExists':"Ability to reward", 
'CONFIG_showOtherSummaries':"Peer outcome visibility",
'CONFIG_showPunishmentId':"Punisher/rewarder anonymity",
'CONFIG_punishmentCost':"Punishment cost",
'CONFIG_punishmentTech':"Punishment effectiveness",
"control_itt_efficiency":'Efficiency under control ("no punishment")'
})

oos_master_featimp_prereg["pct_increase_in_error"] = 100 * (oos_master_featimp_prereg["permuted_performance"] - oos_master_featimp_prereg["baseline"]) / oos_master_featimp_prereg["baseline"]

In [ ]:
oos_master_featimp_prereg.query("model=='E-net'").groupby("feature")["pct_increase_in_error"].mean().sort_values(ascending=False)

feature
Ability to chat                               59.950521
Default contribution vs withdrawal            17.925221
Efficiency under control ("no punishment")    15.937813
"All or Nothing" contributions                14.579931
# of rounds                                   11.492662
Peer outcome visibility                        5.819912
Ability to reward                              4.618892
Visibility of # of rounds                      1.890419
Marginal per capita return                     0.825871
Punishment cost                                0.248291
Punisher/rewarder anonymity                   -0.186912
# of players                                  -0.240426
Punishment effectiveness                      -1.089523
Name: pct_increase_in_error, dtype: float64